In [0]:
%sql
CREATE DATABASE IF NOT EXISTS visagio_rocket_lab.silver;

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

#Particionar pelo ID e ordenar pelo timestamp mais recente
win = Window.partitionBy("id_consumidor").orderBy(F.col("timestamp_ingestion").desc())

df_silver_cons = spark.table("visagio_rocket_lab.bronze.tb_customers") \
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_city").alias("cidade"),
        F.col("customer_state").alias("estado"),
        F.col("timestamp_ingestion")
    ) \
    .withColumn("rn", F.row_number().over(win)) \
    .filter("rn = 1").drop("rn", "timestamp_ingestion")

df_silver_cons.write.mode("overwrite").saveAsTable("visagio_rocket_lab.silver.dim_consumidores")

In [0]:
import pyspark.sql.functions as F

# Carregando as tabelas da Bronze
df_orders = spark.table("visagio_rocket_lab.bronze.tb_orders")
df_items = spark.table("visagio_rocket_lab.bronze.tb_order_items")

df_joined = df_orders.join(df_items, "order_id", "inner")

# Seleção com as colunas corretas para a Gold
df_fat_pedidos = df_joined.select(
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    F.col("product_id").alias("id_produto"), 
    F.col("seller_id"), # NÃO mude o nome aqui para facilitar o join da Gold
    F.col("order_purchase_timestamp").alias("data_compra"),
    F.col("order_delivered_customer_date").alias("data_entrega"),
    F.col("order_estimated_delivery_date").alias("data_entrega_estimada"),
    F.col("price").alias("valor_item"),
    F.col("freight_value").alias("valor_frete"),
    
    # Tradução de Status
    F.when(F.col("order_status") == "delivered", "entregue")
     .when(F.col("order_status") == "canceled", "cancelado")
     .when(F.col("order_status") == "shipped", "enviado")
     .otherwise("outros").alias("status_pedido"),

    # Cálculos de performance
    F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp")).alias("tempo_entrega_real_dias"),
    F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp")).alias("tempo_entrega_estimado_dias")
)

# Coluna de diferença
df_fat_pedidos = df_fat_pedidos.withColumn(
    "diferenca_entrega_dias", 
    F.col("tempo_entrega_real_dias") - F.col("tempo_entrega_estimado_dias")
)

#salvamento
df_fat_pedidos.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("visagio_rocket_lab.silver.fat_pedidos")

print("Sucesso: silver.fat_pedidos atualizada!")

Sucesso: silver.fat_pedidos atualizada!


In [0]:
#Janela genérica para as próximas tabelas
win_senior = Window.partitionBy("id_original").orderBy(F.col("timestamp_ingestion").desc())

#Dimensão Produtos
df_silver_prod = spark.table("visagio_rocket_lab.bronze.tb_products") \
    .select(F.col("product_id").alias("id_produto"), 
            F.col("product_category_name").alias("nome_categoria"),
            F.col("timestamp_ingestion")) \
    .withColumn("id_original", F.col("id_produto")) \
    .withColumn("rn", F.row_number().over(win_senior)) \
    .filter("rn = 1").drop("rn", "timestamp_ingestion", "id_original")

df_silver_prod.write.mode("overwrite").saveAsTable("visagio_rocket_lab.silver.dim_produtos")

#Vendedores
df_silver_vend = spark.table("visagio_rocket_lab.bronze.tb_sellers") \
    .select(F.col("seller_id").alias("id_vendedor"), 
            F.col("seller_city").alias("cidade"),
            F.col("timestamp_ingestion")) \
    .withColumn("id_original", F.col("id_vendedor")) \
    .withColumn("rn", F.row_number().over(win_senior)) \
    .filter("rn = 1").drop("rn", "timestamp_ingestion", "id_original")

df_silver_vend.write.mode("overwrite").saveAsTable("visagio_rocket_lab.silver.dim_vendedores")

In [0]:
%sql
OPTIMIZE visagio_rocket_lab.silver.fat_pedidos ZORDER BY (id_pedido);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 7249600), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775593656323, 1775593656936, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null)"


In [0]:
# 1. Salvar Reviews
spark.table("visagio_rocket_lab.bronze.tb_order_reviews") \
    .write.mode("overwrite") \
    .saveAsTable("visagio_rocket_lab.silver.tb_order_reviews")

# 2. Ajustar Vendedores para bater com o ID da Gold
df_silver_vend = spark.table("visagio_rocket_lab.bronze.tb_sellers") \
    .select(F.col("seller_id"), 
            F.col("seller_city").alias("cidade"),
            F.col("seller_state").alias("estado_vendedor"), 
            F.col("timestamp_ingestion")) \
    .withColumn("rn", F.row_number().over(Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc()))) \
    .filter("rn = 1").drop("rn", "timestamp_ingestion")

df_silver_vend.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("visagio_rocket_lab.silver.dim_vendedores")

print("Tabelas de suporte (Reviews e Vendedores) criadas!")

Tabelas de suporte (Reviews e Vendedores) criadas!
